<h1><center>Analyse de conversations patient-thérapeute sur la santé mentale</h1>

<p><center>La santé mentale est un enjeu majeur de la santé publique. Ce projet vise, dans un pre-
mier temps, à exploiter le jeu de données pour comprendre les types de conversations et leur
structure, puis à analyser les sentiments des échanges afin de détecter les émotions et états
psychologiques des patients. Ensuite, il se concentre sur l’accès et le traitement des conversa-
tions, en résumant les échanges et en identifiant les schémas linguistiques récurrents. Enfin, il
prévoit le développement d’un système question-réponse, capable d’automatiser les interactions
et fournir des réponses.</p>

<center><nav>
    <a href="https://www.kaggle.com/datasets/thedevastator/nlp-mental-health-conversations">Données</a> |
    <a href="https://github.com/cypnet/Analyse-de-conversations-sur-la-sant-mentale-/tree/Logan">Github</a>
</nav>

In [ ]:
# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [7]:
data_path = './train.csv'
df = pd.read_csv(data_path)
print(f"Détails nombre de lignes/colonnes: {df.shape}")
print(f"\nDétails des colonnes: {df.columns.tolist()}")
print(f"\nAperçu des premières lignes:")
df.head()

Détails nombre de lignes/colonnes: (3512, 2)

Détails des colonnes: ['Context', 'Response']

Aperçu des premières lignes:


,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


<h3>Nettoyage des données (pré-traitement)</h3>

<h4>Cellules vides</h4> 

In [8]:
# On doit d'abord vérifier si il y a des cellules vides pour les deux colonnes
print("\nCellules vides (avant traitement): \n",df.isnull().sum())
df = df.dropna(subset=['Response', 'Context'])
print("\nCellules vides (après traitement): \n",df.isnull().sum())


Cellules vides (avant traitement): 
 Context     0
Response    4
dtype: int64

Cellules vides (après traitement): 
 Context     0
Response    0
dtype: int64


<h4>Doublons de cellules</h4>

In [9]:
# On vérifie ensuite si il y a des doublons de cellules
print("\nCellules dupliquées (avant traitement):",df.duplicated().sum())
df = df.drop_duplicates()
print("\nCellules dupliquées (après traitement):",df.duplicated().sum())


Cellules dupliquées (avant traitement): 760

Cellules dupliquées (après traitement): 0


<h4>Ponctuations</h4>
<p>Cela nous permettra de trier les mots plus facilement plus tard, puisque les discussions sont faites de façon orale, les trier n'ont pas vraiment d'utilités</p>

In [10]:
# Fonction qu'on va utiliser pour compter les ponctuations
def count_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponct_counts = {
        'point': txt.count('.'),
        'virgule': txt.count(','),
        'deux points': txt.count(':'),
        'point virgule': txt.count(';'),
        'interogation': txt.count('?'),
        'exclamation': txt.count('!'),
        'guillemet': txt.count('"') + txt.count("'"),
        'tiret': txt.count('-'),
        'slash': txt.count('/')
    }
    return ponct_counts

ponct_data = df['Response'].apply(count_ponctuation)
ponct_df = pd.DataFrame(ponct_data.tolist()) 

print(f"\nPonctuation totale (Avant traitement):\n{ponct_df.sum().sort_values(ascending=False)} \n")

# Pour chaque cellules, on supprimme les ponctuations qui sont comprises dans count_ponctuation en les remplaçant par ''
def remove_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponctuations = ['.', ',', ':', ';', '?', '!', '"', "'", '-','/']
    for p in ponctuations:
        txt = txt.replace(p, '')
    return txt

# On supprime les ponctuations
df['Response'] = df['Response'].apply(remove_ponctuation)
df['Context'] = df['Context'].apply(remove_ponctuation)

ponct_data = df['Response'].apply(count_ponctuation)
ponct_df = pd.DataFrame(ponct_data.tolist()) 

print(f"\nPonctuation totale (Après traitement):\n{ponct_df.sum().sort_values(ascending=False)} \n")


Ponctuation totale (Avant traitement):
point            26633
virgule          19425
guillemet        10001
slash             2728
tiret             2648
interogation      2404
exclamation       1105
deux points        872
point virgule      449
dtype: int64 


Ponctuation totale (Après traitement):
point            0
virgule          0
deux points      0
point virgule    0
interogation     0
exclamation      0
guillemet        0
tiret            0
slash            0
dtype: int64 



In [11]:
# On vérifie que toute les cellules soient du type String et si ce n'est pas le cas (NaN) les remplacer par ''
df['Response'] = df['Response'].fillna('').astype(str)

# On regarde si les cellules Response et Context sont unique, si ce n'est pas le cas alors on supprimme les dupliquées
df = df.drop_duplicates(subset=['Response', 'Context'])

# Pour chaque cellules, on supprimme les ponctuations qui sont comprises dans count_ponctuation en les remplaçant par ''
def remove_ponctuation(txt):
    txt = str(txt) # conversion en String
    ponctuations = ['.', ',', ':', ';', '?', '!', '"', "'", '-','/']
    for p in ponctuations:
        txt = txt.replace(p, '')
    return txt

# On supprime les ponctuations
df['Response'] = df['Response'].apply(remove_ponctuation)
df['Context'] = df['Context'].apply(remove_ponctuation)

<h4>Passer tout les mots en minuscules</h4>
<p>Transformer tout les mots en minuscules nous permettra par la suite de les traiter plus facilement et éviter des situations telle que "toto" soit différent de "Toto".</p>

In [12]:
# compter les majuscules AVANT traitement
df['Response_upper'] = df['Response'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
df['Context_upper'] = df['Context'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

print("\nNombre de majuscules dans Response (avant):", df['Response_upper'].sum())
print("Nombre de majuscules dans Context (avant):", df['Context_upper'].sum())

# transformer le texte en minuscules
df['Response'] = df['Response'].str.lower()
df['Context'] = df['Context'].str.lower()


# compter les majuscules APRES traitement
df['Response_upper_after'] = df['Response'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
df['Context_upper_after'] = df['Context'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

print("\nNombre de majuscules dans Response (après):", df['Response_upper_after'].sum())
print("Nombre de majuscules dans Context (après):", df['Context_upper_after'].sum())


Nombre de majuscules dans Response (avant): 61125
Nombre de majuscules dans Context (avant): 20485

Nombre de majuscules dans Response (après): 0
Nombre de majuscules dans Context (après): 0


<h4>Suppression des espaces insécables (non-breaking space)</h4>
<p>Un espace insécable est définit par le caractère spéciale Unicode <code>'\xa0'</code>. Celui-ci peut se retrouver dans nos données puisque le fichier csv a été récupérée depuis le web, il ne faut donc pas oublier à le retirer.</p>

In [13]:
df['Response'] = df['Response'].str.replace(r'\s+', ' ', regex=True)
df['Context'] = df['Context'].str.replace(r'\s+', ' ', regex=True)

<h4>Tokenisation</h4>
<p>On va maitenant "découper" les mots de chaque cellules en <i>tokens</i>.<br>
Pour pouvoir les définir, on va utiliser la librairie <b>spaCy</b> qui est une version un peut plus moderne que nltk (The Natural Language Toolkit) </p>

<p><i>Il faut au préalable télécharger le modèle pour pouvoir l'utiliser:</i> <code>python3 -m spacy download en_core_web_sm
</code></p>

In [15]:
import spacy
nlp = spacy.load("en_core_web_sm") # On charge spaCy dans python

<h5>Attention !</h5>
<p>Ici on traite une grande quantité de données, tokeniser (si ça se dit...) les mots bêtements va prendre énormément de temps, puisque spacy va devoir exécuter tokenize sur chaque ligne </p>
<p> On va devoir utiliser <code>nlp.pipe()</code> pour traiter les cellules en batch</p>

In [ ]:
docs_response = list(nlp.pipe(df["Response"].astype(str), batch_size=200))
docs_context = list(nlp.pipe(df["Context"].astype(str), batch_size=200))

df['Response_tokens'] = [[token.text for token in doc] for doc in docs_response]
df['Context_tokens'] = [[token.text for token in doc] for doc in docs_context]


In [ ]:
# On affiche le nombre de token total
nb_tokens_response = df['Response_tokens'].apply(len).sum()
nb_tokens_context = df['Context_tokens'].apply(len).sum()
print(f"Nombre total de tokens: {nb_tokens_response + nb_tokens_context}")
print(f"\nNombre total de tokens dans Response: {nb_tokens_response}")
print(f"Nombre total de tokens dans Context: {nb_tokens_context}")

# On affiche les 5 tokens les plus fréquents dans Response et Context pour avoir un apperçu
resp_tokens = [token for tokens in df['Response_tokens'] for token in tokens]
cont_tokens = [token for tokens in df['Context_tokens'] for token in tokens]
print(f"\n5 tokens les plus fréquents dans Response: {Counter(resp_tokens).most_common(5)}")
print(f"5 tokens les plus fréquents dans Context: {Counter(cont_tokens).most_common(5)}")

Nombre total de tokens: 659165

Nombre total de tokens dans Response: 497246
Nombre total de tokens dans Context: 161919

5 tokens les plus fréquents dans Response: [('you', 22082), ('to', 20165), ('the', 13244), ('and', 12961), ('a', 11199)]
5 tokens les plus fréquents dans Context: [('i', 12831), ('and', 5396), ('to', 5177), ('my', 3825), ('a', 3486)]


<p><i>On remarque que la plupart des mots les plus utilisés ne sont que des mots dit "vide"...</i></p>

<h4>Suppression des StopWords</h4>
<p>Des <i>stopwords</i> (ou mots vides en français) sont des mots n'ayant pas de réelle significations. Ses mots sont souvents des adverbes, des pronoms ou encore des mots de liaisons.<br>
<i>source: <a href="https://fr.wikipedia.org/wiki/Mot_vide">Wikipedia: Mot Vide</a> </i> </p>

In [ ]:
df["Response_tokens_no_stop"] = [
    [token.text for token in doc if not token.is_stop and not token.is_punct]
    for doc in docs_response
]

df["Context_tokens_no_stop"] = [
    [token.text for token in doc if not token.is_stop and not token.is_punct]
    for doc in docs_context
]

# On compare les résultats de tokens avec tokens_no_stop
print(f"\n5 tokens les plus fréquents dans Response (avec stopwords): {Counter(resp_tokens).most_common(5)}")
print(f"5 tokens les plus fréquents dans Response (sans stopwords): {Counter([token for tokens in df['Response_tokens_no_stop'] for token in tokens]).most_common(5)}")
print(f"\n5 tokens les plus fréquents dans Context (avec stopwords): {Counter(cont_tokens).most_common(5)}")
print(f"5 tokens les plus fréquents dans Context (sans stopwords): {Counter([token for tokens in df['Context_tokens_no_stop'] for token in tokens]).most_common(5)}")


5 tokens les plus fréquents dans Response (avec stopwords): [('you', 22082), ('to', 20165), ('the', 13244), ('and', 12961), ('a', 11199)]
5 tokens les plus fréquents dans Response (sans stopwords): [('nt', 2163), ('feel', 2114), ('help', 1842), ('like', 1789), ('relationship', 1566)]

5 tokens les plus fréquents dans Context (avec stopwords): [('i', 12831), ('and', 5396), ('to', 5177), ('my', 3825), ('a', 3486)]
5 tokens les plus fréquents dans Context (sans stopwords): [('nt', 1868), ('m', 1327), ('feel', 991), ('like', 918), ('know', 815)]


<h4>Lemmatisation</h4>
<p>Et enfin, il nous faut transformer les mots (tokens) en leurs forme de base, par exemple "running" deviendra "run", ou bien "better" deviendra "good". <br>
Cela permettra d'une part de diminuer drastiquement le nombre de tokens mais aussi de regrouper des mots qui sont similaires</p>

In [24]:
df["Response_clean"] = [
    [
        token.lemma_
        for token in doc
        if not token.is_stop    # stopwords
        and not token.is_punct  # ponctuation
        and not token.is_space  # espaces
    ]
    for doc in docs_response
]

df["Context_clean"] = [
    [
        token.lemma_
        for token in doc
        if not token.is_stop    # stopwords
        and not token.is_punct  # ponctuation
        and not token.is_space  # espaces
    ]
    for doc in docs_context
]

In [25]:
# On affiche maintenant le nombre de tokens après lemmatisation
nb_tokens_response_clean = df['Response_clean'].apply(len).sum()
nb_tokens_context_clean = df['Context_clean'].apply(len).sum()
print(f"Nombre total de tokens après lemmatisation: {nb_tokens_response_clean + nb_tokens_context_clean}")
print(f"\nNombre total de tokens dans Response après lemmatisation: {nb_tokens_response_clean}")
print(f"Nombre total de tokens dans Context après lemmatisation: {nb_tokens_context_clean}")

Nombre total de tokens après lemmatisation: 259775

Nombre total de tokens dans Response après lemmatisation: 201758
Nombre total de tokens dans Context après lemmatisation: 58017


In [27]:
# Apperçus du DataFrame après le traitement
df.head()

,Context,Response,Response_upper,Context_upper,Response_upper_after,Context_upper_after,Response_tokens,Context_tokens,Response_no_stop,Response_tokens_no_stop,Context_tokens_no_stop,Response_clean,Context_clean
0,im going through some things with my feelings ...,if everyone thinks youre worthless then maybe ...,10,10,0,0,"[if, everyone, thinks, you, re, worthless, the...","[i, m, going, through, some, things, with, my,...","[thinks, worthless, maybe, need, find, new, pe...","[thinks, worthless, maybe, need, find, new, pe...","[m, going, things, feelings, barely, sleep, th...","[think, worthless, maybe, need, find, new, peo...","[m, go, thing, feeling, barely, sleep, think, ..."
1,im going through some things with my feelings ...,hello and thank you for your question and seek...,43,10,0,0,"[hello, and, thank, you, for, your, question, ...","[i, m, going, through, some, things, with, my,...","[hello, thank, question, seeking, advice, feel...","[hello, thank, question, seeking, advice, feel...","[m, going, things, feelings, barely, sleep, th...","[hello, thank, question, seek, advice, feeling...","[m, go, thing, feeling, barely, sleep, think, ..."
2,im going through some things with my feelings ...,first thing id suggest is getting the sleep yo...,5,10,0,0,"[first, thing, i, d, suggest, is, getting, the...","[i, m, going, through, some, things, with, my,...","[thing, d, suggest, getting, sleep, need, impa...","[thing, d, suggest, getting, sleep, need, impa...","[m, going, things, feelings, barely, sleep, th...","[thing, d, suggest, get, sleep, need, impact, ...","[m, go, thing, feeling, barely, sleep, think, ..."
3,im going through some things with my feelings ...,therapy is essential for those that are feelin...,14,10,0,0,"[therapy, is, essential, for, those, that, are...","[i, m, going, through, some, things, with, my,...","[therapy, essential, feeling, depressed, worth...","[therapy, essential, feeling, depressed, worth...","[m, going, things, feelings, barely, sleep, th...","[therapy, essential, feel, depressed, worthles...","[m, go, thing, feeling, barely, sleep, think, ..."
4,im going through some things with my feelings ...,i first want to let you know that you are not ...,3,10,0,0,"[i, first, want, to, let, you, know, that, you...","[i, m, going, through, some, things, with, my,...","[want, let, know, feelings, help, change, feel...","[want, let, know, feelings, help, change, feel...","[m, going, things, feelings, barely, sleep, th...","[want, let, know, feeling, help, change, feeli...","[m, go, thing, feeling, barely, sleep, think, ..."


<p><i>On précise bien entendu que toute les étapes peuvent se résumer dans la lemmatisation, le but ici est de détailler toutes les étapes de traitement</i></p>

<h3>Analyse Statistique</h3>

<h4>Nombre de patients uniques</h4>

<p>Il n'y a pas vraiment de façon directe (ou du moins assez fiable) pour définir un patient d'un autre sans un identifiant, la seul façon ici serait de définir une cellule pour un patient et une cellule pour un thérapeute (<b>sans compter les doublons</b>). <br>
Une des suppositions que l'on peut faire serait que chaque cellules patients dupliquée serait un unique patient, celui-ci évoquant la même discussion avec différents thérapeutes. <br>
Cette solution peut fonctionner est très approximative, surtout pour les thérapeutes.</p>

In [29]:
# On compte le nombre de patients uniques dans context_clean
unique_patients = df['Context'].nunique()
print(f"\nNombre de patients uniques: {unique_patients}")


Nombre de patients uniques: 831


<h4>Nombre de thérapeutes uniques</h4>

In [31]:
# On compte le nombre de thérapeutes uniques
unique_therapeutes = df['Response'].nunique()
print(f"Nombre de thérapeutes uniques: {unique_therapeutes}")

Nombre de thérapeutes uniques: 2449


<h4>Distribution des conversations par thérapeute</h4>
<p>On cherche ici combien de conversations peut avoir un thérapeute avec un ou plusieurs patients. Or le problème ici est que, comme au dessus, on a pas d'ID pour les thérapeutes ni pour les patients, donc aucuns moyens de déterminer le nombre de conversations par thérapeutes. </p>

In [32]:
distribution = df['Response'].value_counts()
print(f"\nDistribution des thérapeutes:\n{distribution}")


Distribution des thérapeutes:
Response
hi first and foremost i want to acknowledge your efforts to gain (your) ideal erectile function if the medications are not working and you have taken them as prescribed i would encourage you to seek the help of a sex therapist as the dysfunction may be due to a psychological andor relational issue rather than a physicalmedical one as for your question only you can answer this is it ok are you ok with her sleeping with others have you thought through what this may look like feel like become for you and her opening up a relationship is a choice only the people in the relationship can answer even then the answer may change at any point by either of you i encourage you to also determine what the intention is underneath your telling your girlfriend she could sleep with others be clear with the intention and then together have continuous conversations about the expectations of opening up (ie are there any kinds of sex that is off limits areas of the bo

<h4>Nombre d'échanges moyen par conversation</h4>
<p>Ici, on cherche à voir le nombre de mots moyens par patients/thérapeutes</p>

In [34]:
# Nombre de mots moyens par patients
mean_context_w_stopwords = df['Context'].apply(len).mean()
mean_context_wo_stopwords = df['Context_tokens_no_stop'].apply(len).mean()
mean_context_w_lemmas = df['Context_clean'].apply(len).mean()
print(f"\nNombre de mots moyens par patients (avec stopwords): {mean_context_w_stopwords:.2f}")
print(f"Nombre de mots moyens par patients (sans stopwords): {mean_context_wo_stopwords:.2f}")
print(f"Nombre de mots moyens par patients (avec lemmatisation): {mean_context_w_lemmas:.2f}")

print("\n")

# Nombre de mots moyens par thérapeutes
mean_response_w_stopwords = df['Response'].apply(len).mean()
mean_response_wo_stopwords = df['Response_tokens_no_stop'].apply(len).mean()
mean_response_w_lemmas = df['Response_clean'].apply(len).mean()
print(f"Nombre de mots moyens par thérapeutes (avec stopwords): {mean_response_w_stopwords:.2f}")
print(f"Nombre de mots moyens par thérapeutes (sans stopwords): {mean_response_wo_stopwords:.2f}")
print(f"Nombre de mots moyens par thérapeutes (avec lemmatisation): {mean_response_w_lemmas:.2f}")


Nombre de mots moyens par patients (avec stopwords): 281.26
Nombre de mots moyens par patients (sans stopwords): 21.11
Nombre de mots moyens par patients (avec lemmatisation): 21.11


Nombre de mots moyens par thérapeutes (avec stopwords): 1009.61
Nombre de mots moyens par thérapeutes (sans stopwords): 73.43
Nombre de mots moyens par thérapeutes (avec lemmatisation): 73.42


<h4>Mots les plus utilisés</h4>

In [39]:
# On affiche les 5 tokens les plus fréquents dans Response et Context après lemmatisation pour avoir un apperçu
resp_lemmas = [lemma for lemmas in df['Response_clean'] for lemma in lemmas]
cont_lemmas = [lemma for lemmas in df['Context_clean'] for lemma in lemmas]
print(f"\n5 tokens les plus fréquents dans Response après lemmatisation: {Counter(resp_lemmas).most_common(5)}")
print(f"5 tokens les plus fréquents dans Context après lemmatisation: {Counter(cont_lemmas).most_common(5)}")


5 tokens les plus fréquents dans Response après lemmatisation: [('feel', 3172), ('help', 2182), ('not', 2163), ('relationship', 1948), ('like', 1841)]
5 tokens les plus fréquents dans Context après lemmatisation: [('not', 1868), ('m', 1327), ('feel', 1293), ('like', 956), ('know', 950)]


In [ ]:
# On créer un graphique avec la liste de tout les mots et leurs nombre d'apparition pour les patients(bleu) puis pour les thérapeutes(vert)
all_patient_tokens = [token for tokens in df['Context_clean'] for token in tokens]
all_therapist_tokens = [token for tokens in df['Response_clean'] for token in tokens]
patient_token_counts = Counter(all_patient_tokens)
therapist_token_counts = Counter(all_therapist_tokens)

<h3>Analyse Lexicale</h3>

In [ ]:
# TODO

<h3>Modélisation et extraction de sujets (Topic Modeling)</h3>

In [ ]:
# TODO